# Pokemon Incremental Actor Ingestion

Serverless-compatible notebook task. Reads PokeAPI JSONL from a Unity Catalog Volume and joins random Pokemon actors into the demo Silver/Gold layers.

In [ ]:
from datetime import datetime, timezone
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, IntegerType
from pyspark.sql.window import Window

catalog = 'approvalmax_ai_platform'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'
monitoring_schema = 'monitoring'
volume_name = 'pokemon_ingestion'
input_path = f'/Volumes/{catalog}/{bronze_schema}/{volume_name}/pokeapi_random_pokemon.jsonl'
bronze_table = f'{catalog}.{bronze_schema}.pokemon_actor_raw'
silver_actor_table = f'{catalog}.{silver_schema}.pokemon_actors_current'
silver_bridge_table = f'{catalog}.{silver_schema}.user_pokemon_actor_bridge'
gold_actor_table = f'{catalog}.{gold_schema}.dim_pokemon_actor'
gold_enriched_fact_table = f'{catalog}.{gold_schema}.fact_approval_document_lifecycle_pokemon_actor'
suggestion_table = f'{catalog}.{monitoring_schema}.pokemon_modelling_suggestions'
result_table = f'{catalog}.{monitoring_schema}.great_expectations_results'
run_id = f'pokemon_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
for schema_name in [bronze_schema, silver_schema, gold_schema, monitoring_schema]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}')

pokemon_schema = StructType([
    StructField('source_system', StringType(), True),
    StructField('source_context', StringType(), True),
    StructField('pokemon_id', IntegerType(), True),
    StructField('pokemon_key', StringType(), True),
    StructField('pokemon_name', StringType(), True),
    StructField('base_experience', IntegerType(), True),
    StructField('height', IntegerType(), True),
    StructField('weight', IntegerType(), True),
    StructField('types_json', StringType(), True),
    StructField('abilities_json', StringType(), True),
    StructField('hp', IntegerType(), True),
    StructField('attack', IntegerType(), True),
    StructField('defense', IntegerType(), True),
    StructField('special_attack', IntegerType(), True),
    StructField('special_defense', IntegerType(), True),
    StructField('speed', IntegerType(), True),
    StructField('actor_join_key', StringType(), True),
    StructField('source_url', StringType(), True),
    StructField('ingestion_timestamp', StringType(), True),
    StructField('run_id', StringType(), True),
    StructField('raw_json', StringType(), True),
])

incoming = (
    spark.read.schema(pokemon_schema).json(input_path)
    .withColumn('ingested_at', F.to_timestamp('ingestion_timestamp'))
    .withColumn('_loaded_at', F.current_timestamp())
    .withColumn('_workflow_run_id', F.lit(run_id))
)

spark.sql(f'''CREATE TABLE IF NOT EXISTS {bronze_table} (
  source_system STRING, source_context STRING, pokemon_id INT, pokemon_key STRING, pokemon_name STRING,
  base_experience INT, height INT, weight INT, types_json STRING, abilities_json STRING,
  hp INT, attack INT, defense INT, special_attack INT, special_defense INT, speed INT,
  actor_join_key STRING, source_url STRING, ingestion_timestamp STRING, run_id STRING, raw_json STRING,
  ingested_at TIMESTAMP, _loaded_at TIMESTAMP, _workflow_run_id STRING
) USING DELTA''')

existing_versions = spark.table(bronze_table).select('pokemon_key', 'run_id').distinct()
new_rows = incoming.join(existing_versions, ['pokemon_key', 'run_id'], 'left_anti')
new_count = new_rows.count()
if new_count:
    new_rows.write.format('delta').mode('append').saveAsTable(bronze_table)

bronze = spark.table(bronze_table)
actor_window = Window.partitionBy('pokemon_key').orderBy(F.col('ingested_at').desc_nulls_last(), F.col('_loaded_at').desc())
actors_current = (
    bronze.withColumn('_rn', F.row_number().over(actor_window))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)
actors_current.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_actor_table)

users = spark.table(f'{catalog}.{silver_schema}.users_current').select('user_id', 'company_id')
user_count = users.count()
if user_count == 0:
    raise Exception('Cannot assign Pokemon actors because silver.users_current is empty.')

user_numbered = users.withColumn('user_rank', F.row_number().over(Window.orderBy('user_id')))
actor_numbered = actors_current.withColumn('actor_rank', F.row_number().over(Window.orderBy('pokemon_key')))
bridge = (
    actor_numbered.withColumn('user_rank', ((F.col('actor_rank') - F.lit(1)) % F.lit(user_count)) + F.lit(1))
    .join(user_numbered, 'user_rank', 'inner')
    .select('user_id', 'company_id', 'pokemon_key', 'pokemon_name', 'actor_join_key', F.lit('deterministic_rank_modulo').alias('assignment_rule'), F.current_timestamp().alias('assigned_at'))
)
bridge.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_bridge_table)

spark.sql(f'''CREATE OR REPLACE TABLE {gold_actor_table} AS
SELECT pokemon_key, pokemon_id, pokemon_name, base_experience, height, weight, types_json, abilities_json,
       hp, attack, defense, special_attack, special_defense, speed, actor_join_key, source_url
FROM {silver_actor_table}
WHERE pokemon_key IS NOT NULL''')

spark.sql(f'''CREATE OR REPLACE TABLE {gold_enriched_fact_table} AS
SELECT f.*, b.pokemon_key, a.pokemon_name AS assigned_actor_name, a.types_json AS assigned_actor_types
FROM {catalog}.{gold_schema}.fact_approval_document_lifecycle f
LEFT JOIN {catalog}.{silver_schema}.users_current u ON f.company_id = u.company_id
LEFT JOIN {silver_bridge_table} b ON u.user_id = b.user_id AND u.company_id = b.company_id
LEFT JOIN {gold_actor_table} a ON b.pokemon_key = a.pokemon_key''')

suggestion_rows = [
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'silver', 'object_name': 'pokemon_actors_current', 'suggestion': 'Current-state Silver actor table keyed by pokemon_key.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'silver', 'object_name': 'user_pokemon_actor_bridge', 'suggestion': 'Deterministic rank-modulo bridge to existing users_current.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'gold', 'object_name': 'dim_pokemon_actor', 'suggestion': 'Gold dimension for random Pokemon demo actors.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'gold_enrichment', 'object_name': 'fact_approval_document_lifecycle_pokemon_actor', 'suggestion': 'Non-financial enrichment of lifecycle fact with assigned Pokemon actor.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'dbt', 'object_name': 'pokemon_actor_current_dbt', 'suggestion': 'Generated dbt current-state model after metadata approval.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'pokemon_actor', 'target_layer': 'great_expectations', 'object_name': 'generated_pokemon_actor_checks', 'suggestion': 'Validate pokemon_key/name/raw payload and primary-key grain.', 'human_review_required': 'true'},
]
suggestion_schema = StructType([
    StructField('run_id', StringType(), False),
    StructField('source_context', StringType(), False),
    StructField('target_layer', StringType(), False),
    StructField('object_name', StringType(), False),
    StructField('suggestion', StringType(), False),
    StructField('human_review_required', StringType(), False),
])
spark.createDataFrame(suggestion_rows, schema=suggestion_schema).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(suggestion_table)

spark.sql(f'''CREATE TABLE IF NOT EXISTS {result_table} (
  validation_run_id STRING, expectation_name STRING, status STRING, severity STRING,
  failed_row_count BIGINT, checked_at TIMESTAMP, details STRING
) USING DELTA''')

validation_run_id = f'gx_{run_id}'
expectations = {
    'expect_pokemon_key_not_null': f'SELECT * FROM {silver_actor_table} WHERE pokemon_key IS NULL',
    'expect_pokemon_name_not_null': f'SELECT * FROM {silver_actor_table} WHERE pokemon_name IS NULL',
    'expect_pokemon_raw_json_not_null': f'SELECT * FROM {bronze_table} WHERE raw_json IS NULL',
    'expect_pokemon_actor_grain_unique': f'SELECT pokemon_key, count(*) AS row_count FROM {silver_actor_table} GROUP BY pokemon_key HAVING count(*) > 1',
    'expect_pokemon_bridge_user_not_null': f'SELECT * FROM {silver_bridge_table} WHERE user_id IS NULL',
    'expect_pokemon_gold_enrichment_document_not_null': f'SELECT * FROM {gold_enriched_fact_table} WHERE document_id IS NULL',
}
result_schema = StructType([
    StructField('validation_run_id', StringType(), False),
    StructField('expectation_name', StringType(), False),
    StructField('status', StringType(), False),
    StructField('severity', StringType(), False),
    StructField('failed_row_count', LongType(), False),
    StructField('checked_at', TimestampType(), False),
    StructField('details', StringType(), True),
])
results = []
critical_failures = []
for name, query in expectations.items():
    failed_count = spark.sql(query).count()
    status = 'passed' if failed_count == 0 else 'failed'
    row = {
        'validation_run_id': validation_run_id,
        'expectation_name': name,
        'status': status,
        'severity': 'critical',
        'failed_row_count': failed_count,
        'checked_at': datetime.now(timezone.utc),
        'details': query,
    }
    results.append(row)
    if failed_count:
        critical_failures.append(row)

spark.createDataFrame(results, schema=result_schema).write.format('delta').mode('append').saveAsTable(result_table)
if critical_failures:
    raise Exception('Pokemon Great Expectations-style validation failed: ' + json.dumps(critical_failures, default=str))

print(f'Pokemon ingestion complete. run_id={run_id}, new_rows={new_count}')
